# Customer Churn Classification with a Neural Network

## Project Overview

**Objective:** Predict whether a bank customer will leave using the Churn Modelling dataset.

**Problem type:** Binary classification with `Exited` as the target, where `1` means churned and `0` means stayed.

**Workflow:** Inspect and clean the data, create a stratified 70/15/15 split, fit preprocessing on training data, train one neural network, evaluate the untouched test set, and measure permutation importance.

**Preprocessing:** Numerical features use median imputation and standardization. Categorical features use most-frequent imputation and one-hot encoding.

**Model:** Input → Dense(64) → Dense(32) → Dense(16) → Sigmoid output.

**Evaluation:** Accuracy, Precision, Recall, F1 Score, ROC-AUC, PR-AUC, a classification report, and evaluation curves.

**Interpretability and deployment:** Permutation importance explains global model sensitivity. The trained model and preprocessing artifacts are saved for Streamlit inference.

## Phase 1 — Environment Setup and Data Ingestion

Libraries, reproducible settings, and project-relative paths are prepared before the dataset is loaded.

### Standard Library Imports

`Path`, `json`, and `random` support portable paths, saved metadata, and reproducible seeds.

In [ ]:
# Import project setup tools
from pathlib import Path
import json
import random

### Data and Visualization Imports

Pandas, NumPy, Matplotlib, and Joblib support analysis, visualization, and artifact storage.

In [ ]:
# Import data and plotting tools
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Scikit-learn Imports

Scikit-learn provides leakage-safe preprocessing, stratified splitting, and classification metrics.

In [ ]:
# Import preprocessing and evaluation tools
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


### TensorFlow and Keras

Keras provides the neural-network layers, training workflow, and early stopping.

In [ ]:
# Import the neural-network framework
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

### Reproducibility

Fixed random seeds make the data split and model initialization reproducible.

In [ ]:
# Set reproducible random seeds
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


### Project Paths

Project-relative paths keep the dataset, model artifacts, and figures portable.

In [ ]:
# Resolve project-local paths
roots = [Path.cwd(), Path.cwd().parent]
project_root = next(
    path for path in roots
    if (path / 'data').exists()
)

data_path = project_root / 'data' / 'Churn_Modelling.csv'
models_dir = project_root / 'models'
figures_dir = project_root / 'reports' / 'figures'

models_dir.mkdir(exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)


### Data Loading

The customer churn dataset is loaded and its dimensions are checked before analysis.

In [ ]:
# Load the project dataset
data = pd.read_csv(data_path)

print(f'Dataset path: {data_path}')
print(f'Dataset shape: {data.shape}')


## Phase 2 — Data Understanding

The original data is inspected for structure, quality, target balance, and useful churn relationships before cleaning.

### Dataset Structure

A small sample, column list, and data types confirm the available features and their representation.

In [ ]:
# Inspect columns and data types
display(data.head())
print(data.columns.tolist())
display(data.dtypes.to_frame('data_type'))


### Data Quality Summary

Missing values, unique counts, and data types are reviewed before cleaning decisions are made.

In [ ]:
# Check missing and unique values
quality = pd.DataFrame({
    'missing_values': data.isna().sum(),
    'unique_values': data.nunique(),
    'data_type': data.dtypes.astype(str),
})

display(quality)


### Duplicates and Target Balance

Duplicate counts guide row cleaning, while the `Exited` distribution determines the need for stratified splitting.


In [ ]:
# Check duplicates and target balance
duplicate_count = int(data.duplicated().sum())
target_distribution = data['Exited'].value_counts().sort_index()

print(f'Duplicate rows: {duplicate_count}')
display(target_distribution.to_frame('customers'))


### Numerical Summary

Descriptive statistics are used to check numerical ranges and detect obvious invalid values.

In [ ]:
# Review numerical distributions
display(data.describe().T)


### Customer Churn Patterns

Churn is compared with key numerical and categorical features to identify patterns relevant to modeling.

In [ ]:
# Plot the main churn relationships
figure, axes = plt.subplots(
    3, 3, figsize=(15, 11)
)
axes = axes.ravel()

data['Exited'].value_counts().sort_index().plot.bar(
    ax=axes[0]
)
axes[0].set(
    title='Churn Distribution',
    xlabel='Exited',
    ylabel='Customers',
)

numeric_columns = [
    'Age', 'Balance', 'CreditScore', 'Tenure'
]
for axis, column in zip(axes[1:5], numeric_columns):
    data.boxplot(column=column, by='Exited', ax=axis)
    axis.set_title(f'{column} vs Churn')

category_columns = [
    'Geography', 'Gender',
    'NumOfProducts', 'IsActiveMember',
]
for axis, column in zip(axes[5:], category_columns):
    data.groupby(column)['Exited'].mean().plot.bar(ax=axis)
    axis.set(
        title=f'{column} vs Churn',
        ylabel='Churn Rate',
    )

figure.suptitle(
    'Customer Churn Data Understanding',
    y=1.02,
)
figure.tight_layout()
figure.savefig(
    figures_dir / 'data_understanding.png',
    dpi=160,
)
plt.show()


## Phase 3 — Data Cleaning

Exact duplicates are removed if present. Identifier fields are excluded because they do not represent reusable customer behavior, while missing values remain available for training-only imputation.

### Remove Duplicate Rows

Exact duplicates are removed to prevent repeated records from influencing model training and evaluation.

In [ ]:
# Remove exact duplicate rows
duplicates_before = int(data.duplicated().sum())
data = data.drop_duplicates().copy()
duplicates_after = int(data.duplicated().sum())

print(f'Duplicates before: {duplicates_before}')
print(f'Duplicates after: {duplicates_after}')
print(f'Rows after cleaning: {len(data):,}')


### Remove Identifier Columns

`RowNumber`, `CustomerId`, and `Surname` are removed because they identify records or people rather than customer behavior.

In [ ]:
# Remove identifier-only columns
identifier_columns = [
    'RowNumber', 'CustomerId', 'Surname'
]
target = 'Exited'
modeling_data = data.drop(columns=identifier_columns)

print('Dropped:', identifier_columns)
print('Modeling columns:', modeling_data.columns.tolist())


## Phase 4 — Feature Preparation

Predictors and target are separated before a stratified train, validation, and test split. Learned preprocessing is fitted only on training data to prevent leakage.

### Target and Feature Types

`Exited` is separated from the predictors, and the remaining features are grouped by numerical or categorical type.

In [ ]:
# Separate predictors and target
X = modeling_data.drop(columns=target).copy()
y = modeling_data[target].astype(int).copy()

numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()
categorical_features = X.select_dtypes(
    include='object'
).columns.tolist()

print('Numerical:', numerical_features)
print('Categorical:', categorical_features)


### Train, Validation, and Test Split

A stratified 70/15/15 split preserves churn balance while keeping the test set untouched until final evaluation.

In [ ]:
# Create stratified data splits
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print(X_train.shape, X_validation.shape, X_test.shape)


### Numerical Preprocessing

Numerical features are imputed with the median and scaled using `StandardScaler`.

In [ ]:
# Prepare numerical features
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

numeric_pipeline


### Categorical Preprocessing

Categorical features use most-frequent imputation and `OneHotEncoder`, with unseen categories ignored during inference.

In [ ]:
# Prepare categorical features
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

categorical_pipeline


### Combine Preprocessing Steps

Numerical and categorical transformations are combined in one `ColumnTransformer` for consistent training and inference.

In [ ]:
# Combine preprocessing pipelines
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numerical_features),
    ('categorical', categorical_pipeline, categorical_features),
])

preprocessor


### Fit and Apply Preprocessing

Preprocessing is fitted only on `X_train`, then applied unchanged to validation and test data.

In [ ]:
# Fit preprocessing on training data only
X_train_processed = preprocessor.fit_transform(X_train)
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)


### Keras Input Matrices

Processed matrices use `float32`, and the transformed feature order is retained for artifact validation and deployment.


In [ ]:
# Prepare matrices for Keras
X_train_processed = X_train_processed.astype('float32')
X_validation_processed = X_validation_processed.astype('float32')
X_test_processed = X_test_processed.astype('float32')
feature_names = preprocessor.get_feature_names_out().tolist()

print(X_train_processed.shape, X_validation_processed.shape)
print(X_test_processed.shape, len(feature_names))


## Phase 5 — Neural Network

One neural network is trained with three hidden layers containing 64, 32, and 16 ReLU units. A sigmoid output estimates churn probability, and early stopping restores the best validation weights.

### Neural Network Architecture

The model uses exactly three hidden layers: Dense(64), Dense(32), and Dense(16), followed by one sigmoid output.

In [ ]:
# Build the neural network
model = keras.Sequential(name='customer_churn_nn')
model.add(layers.Input(shape=(X_train_processed.shape[1],)))
model.add(layers.Dense(64, activation='relu', name='dense_64'))
model.add(layers.Dense(32, activation='relu', name='dense_32'))
model.add(layers.Dense(16, activation='relu', name='dense_16'))
model.add(layers.Dense(1, activation='sigmoid', name='output'))


### Model Configuration

The network uses Adam optimization and Binary Crossentropy loss, with Accuracy and AUC tracked during training.

In [ ]:
# Configure neural network training
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)

model.summary()


### Early Stopping

Validation loss controls early stopping, and the best model weights are restored to limit overfitting.

In [ ]:
# Stop when validation loss stops improving
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
)


### Model Training

The model is trained on processed training data while validation data monitors generalization. The test set remains untouched.


In [ ]:
# Train the neural network
history = model.fit(
    X_train_processed,
    y_train,
    validation_data=(
        X_validation_processed,
        y_validation,
    ),
    epochs=120,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1,
)


## Phase 6 — Training Visualization

Training and validation loss and accuracy are compared across epochs to assess convergence and possible overfitting.

### Training History

Loss and accuracy curves show how training performance compares with validation performance over time.

In [ ]:
# Plot training and validation history
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_metrics = [('loss', 'Loss'), ('accuracy', 'Accuracy')]

for axis, (metric, title) in zip(axes, plot_metrics):
    axis.plot(
        history.history[metric],
        label=f'Training {metric}',
    )
    axis.plot(
        history.history[f'val_{metric}'],
        label=f'Validation {metric}',
    )
    axis.set(
        title=f'Training {title}',
        xlabel='Epoch',
        ylabel=title,
    )
    axis.legend()

figure.tight_layout()
figure.savefig(
    figures_dir / 'training_history.png',
    dpi=160,
)
plt.show()


## Phase 7 — Model Evaluation

The final network is evaluated once on untouched test data using classification and ranking metrics. Accuracy is considered alongside Precision, Recall, F1 Score, ROC-AUC, and PR-AUC.

### Test Predictions

Test probabilities are converted to class predictions using the fixed 0.5 decision threshold.

In [ ]:
# Convert probabilities to predicted classes
threshold = 0.5
test_probabilities = model.predict(
    X_test_processed,
    verbose=0,
).ravel()
test_predictions = (
    test_probabilities >= threshold
).astype(int)

print(test_probabilities[:5])


### Classification Metrics

Accuracy, Precision, Recall, F1 Score, ROC-AUC, and PR-AUC provide complementary views of test performance.

In [ ]:
# Calculate complementary test metrics
metrics = {
    'accuracy': accuracy_score(
        y_test, test_predictions
    ),
    'precision': precision_score(
        y_test, test_predictions, zero_division=0
    ),
    'recall': recall_score(
        y_test, test_predictions, zero_division=0
    ),
    'f1_score': f1_score(
        y_test, test_predictions, zero_division=0
    ),
    'roc_auc': roc_auc_score(
        y_test, test_probabilities
    ),
    'pr_auc': average_precision_score(
        y_test, test_probabilities
    ),
}

metrics_table = pd.DataFrame([metrics]).round(4)
display(metrics_table)


### Classification Report

Class-level Precision, Recall, and F1 Score show how performance differs between retained and churned customers.

In [ ]:
# Review performance for both classes
print(classification_report(
    y_test,
    test_predictions,
    zero_division=0,
))


### Evaluation Curves

The confusion matrix shows classification errors, while ROC and precision-recall curves summarize ranking performance.

In [ ]:
# Plot classification evaluation results
figure, axes = plt.subplots(1, 3, figsize=(16, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    ax=axes[0],
)
axes[0].set_title('Confusion Matrix')

false_positive_rate, true_positive_rate, _ = roc_curve(
    y_test,
    test_probabilities,
)
axes[1].plot(
    false_positive_rate,
    true_positive_rate,
    label=f'AUC = {metrics["roc_auc"]:.3f}',
)
axes[1].plot([0, 1], [0, 1], '--', label='No skill')
axes[1].set(
    title='ROC Curve',
    xlabel='False Positive Rate',
    ylabel='True Positive Rate',
)
axes[1].legend()

precision, recall, _ = precision_recall_curve(
    y_test,
    test_probabilities,
)
axes[2].plot(
    recall,
    precision,
    label=f'AP = {metrics["pr_auc"]:.3f}',
)
axes[2].set(
    title='Precision-Recall Curve',
    xlabel='Recall',
    ylabel='Precision',
)
axes[2].legend()

figure.tight_layout()
figure.savefig(
    figures_dir / 'test_evaluation.png',
    dpi=160,
)
plt.show()


## Phase 8 — Feature Importance

Permutation importance is used because neural networks do not expose tree-style feature importance. Features are ranked by the decrease in ROC-AUC after shuffling; the result shows model sensitivity, not causality.

### Permutation Scoring

Each transformed test feature is shuffled independently and the resulting ROC-AUC is measured.

In [ ]:
# Shuffle one transformed test feature
generator = np.random.default_rng(RANDOM_STATE)
baseline_auc = metrics['roc_auc']


def permuted_auc(index):
    shuffled = X_test_processed.copy()
    shuffled[:, index] = generator.permutation(
        shuffled[:, index]
    )
    probabilities = model.predict(
        shuffled,
        verbose=0,
    ).ravel()
    return roc_auc_score(y_test, probabilities)


### Feature Importance Ranking

Transformed features are ranked by their ROC-AUC decrease relative to the unshuffled model.

In [ ]:
# Calculate permutation importance
importance_rows = [
    {
        'feature': name,
        'auc_drop': baseline_auc - permuted_auc(index),
    }
    for index, name in enumerate(feature_names)
]
importance = pd.DataFrame(importance_rows)
importance = importance.sort_values(
    'auc_drop',
    ascending=False,
)

display(importance.head(15))


### Most Important Features

The strongest permutation effects are visualized and saved for the Streamlit model explanation.

In [ ]:
# Plot the most important features
top_features = importance.head(15).sort_values('auc_drop')
figure, axis = plt.subplots(figsize=(8, 6))

axis.barh(
    top_features['feature'],
    top_features['auc_drop'],
)
axis.set(
    title='Permutation Importance',
    xlabel='ROC-AUC decrease',
)

figure.tight_layout()
figure.savefig(
    figures_dir / 'feature_importance.png',
    dpi=160,
)
plt.show()


## Phase 9 — Save Artifacts

The trained model, fitted preprocessing, feature order, metrics, and permutation importance are saved under `models/` for Streamlit inference.

### Save Inference Artifacts

The trained model, fitted preprocessor, and transformed feature order are saved for consistent inference.

In [ ]:
# Save model inference artifacts
model.save(models_dir / 'churn_nn_model.keras')
joblib.dump(
    preprocessor,
    models_dir / 'preprocessor.joblib',
)
joblib.dump(
    feature_names,
    models_dir / 'feature_names.joblib',
)


### Save Evaluation Artifacts

Test metrics and permutation importance are saved so Streamlit can present results without recalculating evaluation.


In [ ]:
# Save metrics and feature importance
metrics['threshold'] = threshold
metrics['test_rows'] = int(len(y_test))
metrics_path = models_dir / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))

importance.to_csv(
    models_dir / 'feature_importance.csv',
    index=False,
)


### Reload Saved Artifacts

The saved model, preprocessor, and feature order are reloaded before deployment validation.

In [ ]:
# Reload the saved inference artifacts
saved_model = keras.models.load_model(
    models_dir / 'churn_nn_model.keras'
)
saved_preprocessor = joblib.load(
    models_dir / 'preprocessor.joblib'
)
saved_names = joblib.load(
    models_dir / 'feature_names.joblib'
)


### Artifact Validation

One untouched test row confirms the saved feature order and verifies that reloaded inference matches the original model.


In [ ]:
# Validate saved artifact compatibility
sample = saved_preprocessor.transform(
    X_test.iloc[:1]
).astype('float32')
prediction = saved_model.predict(
    sample,
    verbose=0,
).ravel()[0]

reloaded_names = (
    saved_preprocessor.get_feature_names_out().tolist()
)
assert saved_names == feature_names == reloaded_names
assert np.isclose(prediction, test_probabilities[0])
assert 0.0 <= prediction <= 1.0
print(f'Reloaded probability: {prediction:.3f}')


# Final Summary

## Data Preparation

Exact duplicates and identifier columns were removed. A stratified 70/15/15 split was created before fitting median imputation, standardization, and one-hot encoding on training data only.

## Model and Training

The single neural network uses Dense(64), Dense(32), and Dense(16) hidden layers with ReLU activations and a sigmoid output. Adam, Binary Crossentropy, and early stopping control training.

## Evaluation

On 1,500 test customers, the model achieved 0.870 Accuracy, 0.796 Precision, 0.485 Recall, 0.603 F1 Score, 0.861 ROC-AUC, and 0.716 PR-AUC. Precision is stronger than Recall, so the fixed 0.5 threshold misses some customers who churn.

## Feature Importance

Permutation importance identified `NumOfProducts`, `Age`, and `IsActiveMember` as the strongest model sensitivities. These associations do not establish causality.

## Deployment

The model, preprocessor, feature names, metrics, and feature-importance table are saved under `models/`. Streamlit applies the saved preprocessor before neural-network inference.

## Limitations

The decision threshold remains fixed at 0.5, and performance may change if the customer population or data distribution shifts.